# Phase 06A.02 — Inner-development checkpoint protocol
Creates a deterministic group-safe split solely from the 1,192 training rows. Frozen validation is not loaded by this notebook.

In [ ]:
import os,sys
from pathlib import Path
PROJECT_ROOT=Path('/workspace/RoadBuddy'); SRC_DIR=PROJECT_ROOT/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *
seed_everything(SEED)

## Configuration

In [ ]:
TRAIN_CSV=PROJECT_ROOT/'data/splits/phase01/train.csv'
DEV_FRACTION=0.20; SEARCH_TRIALS=512
OUTPUT_DIR=PROJECT_ROOT/'data/splits/phase06a_inner'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert TRAIN_CSV.is_file()

## Group-safe deterministic allocation

In [ ]:
train=pd.read_csv(TRAIN_CSV); assert len(train)==EXPECTED_TRAIN_ROWS and train.group_id.nunique()==EXPECTED_TRAIN_GROUPS
train_fit,inner_dev,split_report=group_safe_inner_split(train,dev_fraction=DEV_FRACTION,seed=SEED,search_trials=SEARCH_TRIALS)
train_fit.to_csv(OUTPUT_DIR/'train_fit.csv',index=False); inner_dev.to_csv(OUTPUT_DIR/'inner_dev.csv',index=False)
save_json(OUTPUT_DIR/'train_fit_ids.json',sorted(train_fit.sample_id.astype(str)))
save_json(OUTPUT_DIR/'inner_dev_ids.json',sorted(inner_dev.sample_id.astype(str)))
save_json(OUTPUT_DIR/'inner_split_manifest.json',split_report)
display(split_report)

## Checkpoint and final-retraining contract

In [ ]:
checkpoint_protocol={'selection_data':'inner_dev_only','primary_metric':'accuracy','secondary_metric':'macro_f1','tie_break':['higher_accuracy','higher_macro_f1','lower_optimizer_step'],'patience_evaluations':3,'min_delta':0.0,'eval_steps':8,'save_steps':8,'final_retraining':{'reset_base_model':True,'reset_optimizer_scheduler_rng':True,'training_data':'all_1192_train_rows','stop_at_locked_optimizer_step':True,'frozen_validation_access':False}}
save_json(OUTPUT_DIR/'checkpoint_protocol.json',checkpoint_protocol)
artifacts=[OUTPUT_DIR/'train_fit.csv',OUTPUT_DIR/'inner_dev.csv',OUTPUT_DIR/'inner_split_manifest.json',OUTPUT_DIR/'checkpoint_protocol.json']
write_run_manifest(OUTPUT_DIR/'phase06a_inner_manifest.json',config={'seed':SEED,'dev_fraction':DEV_FRACTION,'search_trials':SEARCH_TRIALS},artifacts=artifacts)
save_json(OUTPUT_DIR/'PHASE06A_02_STATUS.json',{'phase':'06A.02','status':'complete','train_fit_rows':len(train_fit),'inner_dev_rows':len(inner_dev),'group_overlap':0})

## Interpretation constraint
This split selects training horizon/checkpoints only. It does not estimate final baseline performance.